In [3]:
# 5. Analisi delle relazioni tra feature
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from dython.nominal import associations

# Carica il dataset pulito
df_final = pd.read_csv('../data/processed/heloc_dataset_cleanedML.csv')

# Rimuoviamo la colonna target RiskPerformance per concentrarci solo sulle features
if 'RiskPerformance' in df_final.columns:
    df_final = df_final.drop(columns=['RiskPerformance'])

# 1. Identificazione delle colonne categoriche e numeriche
categorical_cols = df_final.select_dtypes(include=['object', 'bool']).columns.tolist()

# Forza le colonne con solo 2 valori (es. flag 0/1) a finire tra le categoriche
for col in df_final.columns:
    if df_final[col].nunique() == 2 and col not in categorical_cols:
        categorical_cols.append(col)

# Aggiorna le numeriche escludendo quelle appena identificate come categoriche
numeric_cols = [col for col in df_final.columns if col not in categorical_cols]

dataset_name = "heloc_ML"
output_dir = "../data/processed/analysisML/"
os.makedirs(output_dir, exist_ok=True)

# Eseguiamo l'associazione completa usando dython
assoc = associations(
    df_final, 
    nominal_columns=categorical_cols,
    numerical_columns=numeric_cols,
    nom_nom_assoc='cramer',
    num_num_assoc='pearson',
    compute_only=True,
    plot=False
)

corr_matrix = assoc['corr']

# 1. Pearson (Numerico & Numerico)
pearson_matrix = corr_matrix.loc[numeric_cols, numeric_cols]
pearson_matrix.to_csv(os.path.join(output_dir, f"{dataset_name}_pearson_matrix.csv"))

plt.figure(figsize=(12, 10))
sns.heatmap(pearson_matrix, annot=False, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Pearson Correlation (Num & Num)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_pearson_heatmap.png"))
plt.close()

# 2. V di Cramér (Categorico & Categorico)
if len(categorical_cols) > 0:
    cramersv_matrix = corr_matrix.loc[categorical_cols, categorical_cols]
    cramersv_matrix.to_csv(os.path.join(output_dir, f"{dataset_name}_cramersv_matrix.csv"))

    plt.figure(figsize=(10, 8))
    sns.heatmap(cramersv_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
    plt.title("Cramér's V (Cat & Cat)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{dataset_name}_cramersv_heatmap.png"))
    plt.close()

# 3. Rapporto di Correlazione η (Categorico & Numerico)
if len(categorical_cols) > 0 and len(numeric_cols) > 0:
    eta_matrix = corr_matrix.loc[categorical_cols, numeric_cols]
    eta_matrix.to_csv(os.path.join(output_dir, f"{dataset_name}_eta_matrix.csv"))

    plt.figure(figsize=(12, 6))
    sns.heatmap(eta_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
    plt.title("Correlation Ratio Eta (Cat & Num)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{dataset_name}_eta_heatmap.png"))
    plt.close()

print(f"Analisi completata! Categoriche/bivalenti rilevate: {len(categorical_cols)}, Numeriche rilevate: {len(numeric_cols)}")
print("Matrici e heatmap salvate con successo nella cartella", output_dir)


Analisi completata! Categoriche/bivalenti rilevate: 11, Numeriche rilevate: 23
Matrici e heatmap salvate con successo nella cartella ../data/processed/analysisML/


In [4]:

# Analisi della relazione tra feature e target
import pandas as pd
import numpy as np
import os
from dython.nominal import associations

df_final = pd.read_csv('../data/processed/heloc_dataset_cleanedML.csv')

target_col = 'RiskPerformance'
dataset_name = 'heloc_ML'
output_dir = '../data/processed/analysisML/'
os.makedirs(output_dir, exist_ok=True)

# 1. Identificazione delle colonne categoriche e numeriche (target compreso pèr darlo in pasto a dython)
categorical_cols = df_final.select_dtypes(include=['object', 'bool']).columns.tolist()

# Come nello Step 5, forziamo le costanti booleane 0/1 a esser trattate come Categoriche
for col in df_final.columns:
    if df_final[col].nunique() == 2 and col not in categorical_cols:
        categorical_cols.append(col)

numeric_cols = [col for col in df_final.columns if col not in categorical_cols]

target_is_categorical = target_col in categorical_cols

# Generiamo tutte le associazioni (dython sa già fare V di Cramér, Eta e Pearson nel background combinandole correttamente)
assoc = associations(
    df_final, 
    nominal_columns=categorical_cols,
    numerical_columns=numeric_cols,
    nom_nom_assoc='cramer',
    num_num_assoc='pearson',
    compute_only=True,
    plot=False
)

corr_matrix = assoc['corr']

# Estraiamo la riga delle sole relazioni verso il target
target_associations = corr_matrix.loc[target_col].drop(target_col)

results = []

for feature in target_associations.index:
    valore = target_associations[feature]
    
    is_categorical_feature = (feature in categorical_cols)
    
    # Determiniamo testualmente la metrica utilizzata in base al tipo
    if target_is_categorical:
        if is_categorical_feature:
            metrica = "V di Cramér"
            tipo_feature = "Categorica"
        else:
            metrica = "η (Correlation Ratio Eta)"
            tipo_feature = "Numerica"
    else:
        if is_categorical_feature:
            metrica = "η (Correlation Ratio Eta)"
            tipo_feature = "Categorica"
        else:
            metrica = "Pearson"
            tipo_feature = "Numerica"
            
    results.append({
        'dataset': dataset_name,
        'feature': feature,
        'tipo_feature': tipo_feature,
        'target': target_col,
        'metrica_usata': metrica,
        'valore_associazione': valore
    })

df_results = pd.DataFrame(results)

# Ordinamento decrescente in base all'importanza (al valore assoluto dell'associazione: da più influente a meno influente)
df_results['valore_assoluto'] = df_results['valore_associazione'].abs()
df_results = df_results.sort_values(by='valore_assoluto', ascending=False).drop(columns=['valore_assoluto'])

# Salvataggio
output_file = os.path.join(output_dir, f"{dataset_name}_feature_target_associazioni.csv")
df_results.to_csv(output_file, index=False)

print(f"Tabella salvata come: {output_file}\n")
print("Top 10 feature (primo ranking di importanza):")
display(df_results.head(10))



/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_56985/268100102.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_final.select_dtypes(include=['object', 'bool']).columns.tolist()


Tabella salvata come: ../data/processed/analysisML/heloc_ML_feature_target_associazioni.csv

Top 10 feature (primo ranking di importanza):


,dataset,feature,tipo_feature,target,metrica_usata,valore_associazione
0,heloc_ML,ExternalRiskEstimate,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.460381
17,heloc_ML,NetFractionRevolvingBurden,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.335938
22,heloc_ML,PercentTradesWBalance,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.276454
7,heloc_ML,PercentTradesNeverDelq,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.257591
3,heloc_ML,AverageMInFile,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.246518
9,heloc_ML,MaxDelq2PublicRecLast12M,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.236895
23,heloc_ML,MSinceMostRecentDelq_never,Categorica,RiskPerformance,V di Cramér,0.230241
21,heloc_ML,NumBank2NatlTradesWHighUtilization,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.217682
10,heloc_ML,MaxDelqEver,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.216894
1,heloc_ML,MSinceOldestTradeOpen,Numerica,RiskPerformance,η (Correlation Ratio Eta),0.205119
